In [1]:
import torch
import torch.nn as nn

In [6]:
# A simple Structure of Encoder only

class Attention(nn.Module):
    def __init__(self, embed_size):
        super(Attention, self).__init__()
        self.embed_size = embed_size
        self.query = nn.Linear(embed_size, embed_size)
        self.key = nn.Linear(embed_size, embed_size)
        self.value = nn.Linear(embed_size, embed_size)
        
    def forward(self, x):
        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)

        attention_score = torch.matmul(Q, K.transpose(-2, -1)) / (self.embed_size ** 0.5)
        attention_weights = torch.softmax(attention_score, dim=-1)
        out = torch.matmul(attention_weights, V)
        return out
    
# this is attention layer where we have query , key and value of the same shape as input x which is (batch_size, seq_length, dimensions) and we are calculating attention score by multiplying query and key and then applying softmax to get attention weights and finally multiplying attention weights with value to get the output of attention layer.
    

In [12]:
class Encoder_only(nn.Module):
    def __init__(self, embed_size):
        super(Encoder_only,self).__init__()
        self.attention = Attention(embed_size)
        self.norm1 = nn.LayerNorm(embed_size)
        self.norm2 = nn.LayerNorm(embed_size)
        self.feed_Forward = nn.Sequential(
            nn.Linear(embed_size, embed_size*4),
            nn.ReLU(),
            nn.Linear(embed_size*4, embed_size)
        )
    def forward(self,x):
        attention_out = self.attention(x)
        x = self.norm1(attention_out + x)
        feed_forward = self.feed_Forward(x)
        out = self.norm2(feed_forward + x)
        return out

In [13]:
dummy_input = torch.rand(2, 5, 10) # (batch_size, seq_length, dimensions)
model = Encoder_only(embed_size=10)
output = model(dummy_input)
print(output.shape) # should be (2, 5, 10)

torch.Size([2, 5, 10])
